# Search Laboratory: BFS and A*

This notebook implements uninformed BFS, informed A*, correctness tests, search-effort metrics, and a heuristic comparison.

In [ ]:

from collections import deque
from dataclasses import dataclass
from itertools import count
import heapq
import math

WAREHOUSE = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
]

MOVES = (
    ("Up", -1, 0),
    ("Down", 1, 0),
    ("Left", 0, -1),
    ("Right", 0, 1),
)


@dataclass
class SearchResult:
    algorithm: str
    positions: list[tuple[int, int]]
    actions: list[str]
    states_expanded: int

    @property
    def found(self):
        return bool(self.positions)

    @property
    def path_length(self):
        return len(self.actions) if self.found else None


def parse_grid(grid):
    if not grid or len({len(row) for row in grid}) != 1:
        raise ValueError("Grid must be non-empty and rectangular.")

    start = goal = None
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            if value == "S":
                if start is not None:
                    raise ValueError("Multiple start states found.")
                start = (r, c)
            elif value == "G":
                if goal is not None:
                    raise ValueError("Multiple goal states found.")
                goal = (r, c)

    if start is None or goal is None:
        raise ValueError("Grid must contain exactly one S and one G.")

    return start, goal


def successors(grid, state):
    rows, cols = len(grid), len(grid[0])
    r, c = state

    for action, dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if (
            0 <= nr < rows
            and 0 <= nc < cols
            and grid[nr][nc] != "#"
        ):
            yield action, (nr, nc)


def reconstruct(parent, goal, algorithm, expanded):
    positions, actions = [], []
    state = goal

    while state is not None:
        positions.append(state)
        previous, action = parent[state]
        if action is not None:
            actions.append(action)
        state = previous

    return SearchResult(
        algorithm=algorithm,
        positions=list(reversed(positions)),
        actions=list(reversed(actions)),
        states_expanded=expanded,
    )


def bfs(grid):
    start, goal = parse_grid(grid)
    frontier = deque([start])
    parent = {start: (None, None)}
    expanded = 0

    while frontier:
        state = frontier.popleft()

        if state == goal:
            return reconstruct(parent, goal, "BFS", expanded)

        expanded += 1
        for action, nxt in successors(grid, state):
            if nxt not in parent:
                parent[nxt] = (state, action)
                frontier.append(nxt)

    return SearchResult("BFS", [], [], expanded)


def astar(grid, heuristic, name="A*"):
    start, goal = parse_grid(grid)
    serial = count()

    # Heap entry: (f = g + h, tie breaker, g, state)
    frontier = [(heuristic(start, goal), next(serial), 0, start)]
    best_g = {start: 0}
    parent = {start: (None, None)}
    expanded = 0

    while frontier:
        _, _, g, state = heapq.heappop(frontier)

        # Ignore an outdated heap entry if a cheaper path was found later.
        if g != best_g.get(state):
            continue

        if state == goal:
            return reconstruct(parent, goal, name, expanded)

        expanded += 1
        for action, nxt in successors(grid, state):
            candidate_g = g + 1

            if candidate_g < best_g.get(nxt, math.inf):
                best_g[nxt] = candidate_g
                parent[nxt] = (state, action)
                f = candidate_g + heuristic(nxt, goal)
                heapq.heappush(frontier, (f, next(serial), candidate_g, nxt))

    return SearchResult(name, [], [], expanded)


def manhattan(state, goal):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


def euclidean(state, goal):
    return math.hypot(state[0] - goal[0], state[1] - goal[1])


def validate_path(grid, result):
    start, goal = parse_grid(grid)

    if not result.found:
        return False

    assert result.positions[0] == start
    assert result.positions[-1] == goal
    assert len(result.positions) == len(result.actions) + 1

    for a, b in zip(result.positions, result.positions[1:]):
        assert abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1
        assert grid[b[0]][b[1]] != "#"

    return True


def run_required_tests():
    trivial = [
        "#####",
        "#SG##",
        "#####",
    ]

    no_solution = [
        "#######",
        "#S....#",
        "###.###",
        "#...#G#",
        "#######",
    ]

    alternatives = [
        "#######",
        "#S....#",
        "#.....#",
        "#....G#",
        "#######",
    ]

    original = astar(WAREHOUSE, manhattan, "A* Manhattan")
    one_step = astar(trivial, manhattan, "A* trivial")
    impossible = astar(no_solution, manhattan, "A* no solution")
    multiple = astar(alternatives, manhattan, "A* alternatives")

    assert validate_path(WAREHOUSE, original)
    assert one_step.path_length == 1 and validate_path(trivial, one_step)
    assert not impossible.found
    assert validate_path(alternatives, multiple)
    assert multiple.path_length == bfs(alternatives).path_length

    return original


def main():
    astar_result = run_required_tests()
    bfs_result = bfs(WAREHOUSE)

    assert validate_path(WAREHOUSE, bfs_result)
    assert bfs_result.path_length == astar_result.path_length

    print("Required correctness tests: PASS")
    print("\nBFS versus A*")
    print(f"{'Algorithm':<18}{'Path length':<14}{'Expanded'}")
    for result in (bfs_result, astar_result):
        print(f"{result.algorithm:<18}{result.path_length:<14}{result.states_expanded}")

    heuristic_specs = [
        ("h = 0", lambda state, goal: 0),
        ("Manhattan", manhattan),
        ("Euclidean", euclidean),
        ("2 x Manhattan", lambda state, goal: 2 * manhattan(state, goal)),
    ]

    print("\nHeuristic experiment")
    print(f"{'Heuristic':<18}{'Path length':<14}{'Expanded'}")
    for name, heuristic in heuristic_specs:
        result = astar(WAREHOUSE, heuristic, name)
        assert result.found and validate_path(WAREHOUSE, result)
        print(f"{name:<18}{result.path_length:<14}{result.states_expanded}")

    print("\nOriginal A* path:")
    print(astar_result.positions)




In [ ]:
main()


# Search Lab — Short Notes / Answers

## Algorithms implemented

- **BFS:** blind/uninformed search using a FIFO queue.
- **A\*:** informed search using `f(n) = g(n) + h(n)`.
- Both reconstruct the actual route by storing each discovered state's parent and action.
- `states_expanded` is recorded so search effort can be compared.

## Why BFS is a valid baseline

Every move in the warehouse costs exactly one step. Therefore BFS explores nodes in nondecreasing path depth and returns a shortest path whenever one exists.

## Why Manhattan distance is suitable

For four-direction movement,

`h(n) = |row_n - row_goal| + |col_n - col_goal|`.

Ignoring obstacles can only make the estimated route shorter than or equal to the real route, so Manhattan distance is admissible. It is also consistent for unit-cost Up/Down/Left/Right movement.

## Required correctness tests

The code checks:

1. the original warehouse has a legal solution;
2. a one-step `S -> G` map returns length `1`;
3. an explicitly blocked map returns no solution;
4. a map with alternative routes returns a legal shortest path, verified against BFS.

A returned route is additionally checked to:
- start at `S`;
- end at `G`;
- move one grid square per action;
- never enter `#`.

## BFS vs A*

Both should produce an optimal path when A* uses an admissible heuristic. A* can expand fewer states when the heuristic is informative, although on some obstacle layouts the chosen heuristic may offer little advantage.

## Heuristic experiment

- **`h = 0`:** A* becomes uniform-cost search; with unit costs this behaves like BFS in terms of optimality.
- **Manhattan:** admissible and well matched to four-direction movement.
- **Euclidean:** also admissible here, but usually less informed than Manhattan for grid movement.
- **`2 × Manhattan`:** can overestimate the true remaining cost, so it is not admissible and optimality is no longer guaranteed even if it happens to find an optimal path on this map.

## Reflection

The important validation is not merely that a search function returns something. The route must satisfy the transition rules and obstacle constraints, and optimality claims should be checked against an independent baseline such as BFS.
